# ATML PA3 — Task 1 evaluation

Attach `atml-pa3-code` **and** the output(s) of the training notebook versions (Add Input →
Your Work). GPU T4 x2, Internet on, then *Save & Run All*. Conditions whose training export
is missing are skipped, so this can run as soon as some runs finish; rerunning skips
conditions that already have results.

In [ ]:
# --- locate code (Kaggle dataset "atml-pa3-code" containing atml-pa3-code.zip, or Colab Drive) ---
import glob, os, shutil, subprocess, sys, zipfile
ON_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if ON_KAGGLE else "/content/drive/MyDrive/atml-pa3"
if not ON_KAGGLE:
    from google.colab import drive; drive.mount("/content/drive")
os.makedirs(WORK, exist_ok=True)
REPO = "/tmp/ATML-PA3"           # code lives outside the saved output
zips = glob.glob("/kaggle/input/**/atml-pa3-code.zip", recursive=True) + glob.glob(f"{WORK}/atml-pa3-code.zip")
# Kaggle usually unpacks an uploaded zip, so also accept an already-extracted copy
dirs = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/task1/train.py", recursive=True)]
shutil.rmtree(REPO, ignore_errors=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall(REPO)
else:
    assert dirs, "upload outputs/atml-pa3-code.zip as a Kaggle dataset (or to Drive/atml-pa3/)"
    shutil.copytree(dirs[0], REPO)
print("code from", zips[:1] or dirs[:1])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "hf_xet"], check=True)
os.chdir(REPO)
os.makedirs(f"{WORK}/outputs", exist_ok=True)
if not os.path.exists("outputs"):
    os.symlink(f"{WORK}/outputs", "outputs")
import torch; print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
CONDITIONS = ["baseline", "full_u625", "full", "l2sp", "lora4", "lora16", "oplora16"]
# restore training exports + any earlier eval results from attached notebook outputs
for pat in ["/kaggle/input/*/outputs/task1/*", "/kaggle/input/*/outputs/task1_eval_models/*",
            "/kaggle/input/*/outputs/results_task1/*"]:
    for prev in glob.glob(pat) if ON_KAGGLE else []:
        dst = os.path.join("outputs", os.path.relpath(prev, os.path.dirname(os.path.dirname(prev))))
        if not os.path.exists(dst) and not prev.endswith("/resume"):
            print("restoring", prev); shutil.copytree(prev, dst, ignore=shutil.ignore_patterns("resume"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lm_eval==0.4.13", "langdetect", "immutabledict"], check=True)

In [ ]:
# try vLLM (much faster GSM8K generation); fall back to HF if it does not work on this GPU
BACKEND = "hf"
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
test = subprocess.run([sys.executable, "-c",
    "from vllm import LLM, SamplingParams; m=LLM('Qwen/Qwen3-0.6B', dtype='float16', max_model_len=2048, gpu_memory_utilization=0.5);"
    "print(m.generate(['1+1='], SamplingParams(max_tokens=5))[0].outputs[0].text)"], capture_output=True, text=True)
print(test.stdout[-500:], test.stderr[-1500:])
if test.returncode == 0:
    BACKEND = "vllm"
print("BACKEND =", BACKEND)

In [ ]:
def have(cond):
    src = {"full_u625": "full/export_u625", "full": "full/export_u1250", "l2sp": "l2sp/export_u1250",
           "lora4": "lora4/export_u1250", "lora16": "lora16/export_u1250", "oplora16": "oplora16/export_u1250"}
    return cond == "baseline" or os.path.exists(f"outputs/task1/{src[cond]}")
ready = [c for c in CONDITIONS if have(c)]
print("ready:", ready)
subprocess.run([sys.executable, "-m", "task1.prepare_eval", "--only", *ready], check=True)

In [ ]:
RES = "outputs/results_task1"
todo = [c for c in ready if not os.path.exists(f"{RES}/{c}/qualitative.json")]
print("evaluating", todo)
for c in todo:   # one condition at a time so partial progress is kept
    subprocess.run([sys.executable, "-m", "task1.evaluate", "--backend", BACKEND, "--out", RES,
                    "--conditions", c], check=True)

In [ ]:
# eval models are rebuilt cheaply; drop them so the saved output stays small
shutil.rmtree("outputs/task1_eval_models", ignore_errors=True)
!du -sh outputs/*